In [5]:
# %%
import warnings
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

BASE_PATH = '/Users/egorilin/Desktop/COMA'
BASE = Path(BASE_PATH)

RANDOM_STATE = 42
TEST_SIZE = 0.2

CB_KW = dict(iterations=500, learning_rate=0.05, depth=6,
             verbose=False, random_seed=RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]

# 'biocides' — 12 фич (без диполей); 'conc' — 15 фич (с диполями)
DATASET_MODE = 'conc'

FEATURE_COLS_FULL = [
    'Molecular weight', 'LogP', 'mmff94', 'gasteiger', 'eem2015bm',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS_BIO = [
    'Molecular weight', 'LogP',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS = FEATURE_COLS_BIO if DATASET_MODE == 'biocides' else FEATURE_COLS_FULL
DIPOLE_COLS = ['mmff94', 'gasteiger', 'eem2015bm']

DESC_OUT_DIR = BASE / 'descriptors_test_predictions'
EMB_OUT_DIR  = BASE / 'embeddings_test_predictions'
DESC_OUT_DIR.mkdir(exist_ok=True)
EMB_OUT_DIR.mkdir(exist_ok=True)

print(f'Dataset mode: {DATASET_MODE}  |  {len(FEATURE_COLS)} фич')
print(f'Output dirs:\n  {DESC_OUT_DIR}\n  {EMB_OUT_DIR}')

Device: cpu
Dataset mode: conc  |  15 фич
Output dirs:
  /Users/egorilin/Desktop/COMA/descriptors_test_predictions
  /Users/egorilin/Desktop/COMA/embeddings_test_predictions


In [6]:
# %%
target = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')
desc_df = pd.read_excel(f'{BASE_PATH}/Data_molformer_conc.xlsx')
data_drop = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(
    ['Unnamed: 0'], axis=1
)
dipoles_df = pd.read_csv(f'{BASE_PATH}/Train_dipole_moments.csv')

print(f'target:     {target.shape}')
print(f'desc_df:    {desc_df.shape}')
print(f'data_drop:  {data_drop.shape}')
print(f'dipoles_df: {dipoles_df.shape}  | cols: {list(dipoles_df.columns)}')

# --- Автоопределение ключа мерджа ---
candidates = ['SMILES', 'smiles', 'Smiles', 'canonical_smiles',
              'Name', 'name', 'Compound', 'compound', 'ID', 'id']
merge_key = None
for k in candidates:
    if k in desc_df.columns and k in dipoles_df.columns:
        merge_key = k
        break

if merge_key is None:
    common = (set(desc_df.columns) & set(dipoles_df.columns)) - set(DIPOLE_COLS) - set(TARGET_COLUMNS)
    raise ValueError(
        f'Не нашёл ключ для мерджа. Общие колонки: {common}. '
        f'desc_df: {list(desc_df.columns)[:20]}; '
        f'dipoles_df: {list(dipoles_df.columns)}'
    )
print(f'\n✓ Ключ для мерджа: "{merge_key}"')

have_dipoles = [c for c in DIPOLE_COLS if c in dipoles_df.columns]
miss_dipoles = [c for c in DIPOLE_COLS if c not in dipoles_df.columns]
print(f'  Диполи в файле: {have_dipoles}')
if miss_dipoles:
    print(f'  ⚠️  Нет в файле диполей: {miss_dipoles}')

desc_df = desc_df.drop(columns=[c for c in DIPOLE_COLS if c in desc_df.columns])
desc_df = desc_df.merge(
    dipoles_df[[merge_key] + have_dipoles].drop_duplicates(subset=[merge_key]),
    on=merge_key, how='left'
)

print('\nПокрытие диполей после мерджа:')
for c in have_dipoles:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Проверки ---
missing_desc = [c for c in FEATURE_COLS if c not in desc_df.columns]
if missing_desc:
    raise ValueError(f'Нет колонок в desc_df: {missing_desc}')

for c in TARGET_COLUMNS:
    if c not in desc_df.columns:
        raise ValueError(f'Нет таргета в desc_df: {c}')
    if c not in target.columns:
        raise ValueError(f'Нет таргета в target: {c}')

print('\n✓ Все нужные колонки на месте')
print(f'  FEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}')

target:     (108, 10)
desc_df:    (82, 786)
data_drop:  (108, 768)
dipoles_df: (108, 4)  | cols: ['Unnamed: 0', 'mmff94', 'gasteiger', 'eem2015bm']


ValueError: Не нашёл ключ для мерджа. Общие колонки: {'Unnamed: 0'}. desc_df: ['Unnamed: 0', 'General formula', 'Head', 'Spacer', 'Tail', 'Counterion', 'Molecular weight', 'Gram_aureus', 'Gram_coli', 'LogP', 'Ecoli_MIC', 'S. aureus ATCC 43300 Activity MIC, мг/л', 'S. aureus ATCC 43300 Activity MBC, мг/л', 'E. coli ATCC 25922 Activity MIC, мг/л', 'E. coli ATCC 25922 Activity MBC, мг/л', 'mmff94_dipole', 'gasteiger_dipole', 'eem2015bm_dipole', 'Molformer_0', 'Molformer_1']; dipoles_df: ['Unnamed: 0', 'mmff94', 'gasteiger', 'eem2015bm']

In [ ]:
# %%
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 128],
                 projection_dim=256, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
        )

    def forward(self, x, return_projection=True):
        features = self.encoder(x)
        if return_projection:
            return features, self.projection_head(features)
        return features


encoder = AdvancedContrastiveEncoder(
    input_dim=768, hidden_dims=[512, 384, 128],
    projection_dim=128, dropout=0.3,
).to(device)
encoder.load_state_dict(
    torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth', map_location=device)
)
encoder.eval()

with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
    scaler = pickle.load(f)

data_scaled = scaler.transform(data_drop)
X_tensor = torch.FloatTensor(data_scaled).to(device)
with torch.no_grad():
    molecular_embeddings = encoder(X_tensor, return_projection=False).cpu().numpy()

print(f'✓ molecular_embeddings: {molecular_embeddings.shape}')

RuntimeError: Error(s) in loading state_dict for AdvancedContrastiveEncoder:
	size mismatch for encoder.8.weight: copying a param with shape torch.Size([256, 384]) from checkpoint, the shape in current model is torch.Size([128, 384]).
	size mismatch for encoder.8.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.weight: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.running_mean: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.running_var: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for